# Preprocessing QA (ST-048)

Before/after views of stages S2–S7 for the M2 review. Panels are produced by `scripts/preprocess_qa.py`, which calls the same backend functions as inference.

**Review checklist (tick in the M2 demo notes):**
- [ ] Red bottom-track line follows the first seabed return on both sides; green recorded altitude is shown for comparison
- [ ] Gain-corrected image has no strong brightness trend across range; port and starboard look balanced
- [ ] Ground-range image: nadir band removed, straight features stay straight, no stretching along the track
- [ ] Lee channel is smoother than raw; texture channel highlights edges
- [ ] Dropout and high-motion masks cover the visibly bad rows only
- [ ] Cleaned track follows the raw fixes without jumps

In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "preprocess_qa.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
from preprocess_qa import run_qa

SOURCES = sorted((ROOT / "data/raw/usgs/grandbay_2015-315-FA").glob("*.xtf"))
OUT = ROOT / "data/results/qa"
SOURCES

In [ ]:
PANELS = ["1_raw_bottom", "2_gain", "3_ground", "4_channels", "5_masks", "6_track"]
for source in SOURCES:
    if source.stat().st_size < 1_000_000:
        continue  # skip very short lines
    summary = run_qa(source, OUT)
    display(Markdown(f"## {source.name}"))
    display(Markdown("```json\n" + json.dumps(summary, indent=2) + "\n```"))
    for panel in PANELS:
        path = OUT / source.stem / f"{panel}.png"
        if path.exists():
            display(Markdown(f"**{panel}**"))
            display(Image(filename=str(path), width=900))